# Instalando a bibloteca nba_api

In [ ]:
pip install --upgrade nba-api

# Importando biblotecas e funções de nba_api


In [ ]:
from nba_api.stats.endpoints import leaguedashteamstats
from nba_api.stats.endpoints import commonplayoffseries
from nba_api.stats.endpoints import leaguedashteamshotlocations
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguedashplayerstats
import pandas as pd
import numpy as np


# Filtrando os times campeões

In [ ]:
seasons = pd.Series(["2015-16","2016-17","2017-18","2018-19","2019-20","2020-21",
           "2021-22","2022-23","2023-24","2024-25"])

teams_Series = ["CLEVELAND CAVALIERS", "GOLDEN STATE WARRIORS",
                      "GOLDEN STATE WARRIORS", "TORONTO RAPTORS","LOS ANGELES LAKERS",
                      "MILWAUKEE BUCKS", "GOLDEN STATE WARRIORS", "BOSTON CELTICS",
                        "DENVER NUGGETS", "OKLAHOMA CITY THUNDER"]

df_last_10_champions = pd.DataFrame(index=seasons)
df_last_10_champions["TEAMS"] = teams_Series



# Filtrando todos os times por id e por nome


In [ ]:
nba_teams = [
    # Conferência Leste
    "ATLANTA HAWKS", "BOSTON CELTICS", "BROOKLYN NETS", "CHARLOTTE HORNETS",
    "CHICAGO BULLS", "CLEVELAND CAVALIERS", "DETROIT PISTONS", "INDIANA PACERS",
    "MIAMI HEAT", "MILWAUKEE BUCKS", "NEW YORK KNICKS", "ORLANDO MAGIC",
    "PHILADELPHIA 76ERS", "TORONTO RAPTORS", "WASHINGTON WIZARDS",

    # Conferência Oeste
    "DALLAS MAVERICKS", "DENVER NUGGETS", "GOLDEN STATE WARRIORS", "HOUSTON ROCKETS",
    "LOS ANGELES CLIPPERS", "LOS ANGELES LAKERS", "MEMPHIS GRIZZLIES",
    "MINNESOTA TIMBERWOLVES", "NEW ORLEANS PELICANS", "OKLAHOMA CITY THUNDER",
    "PHOENIX SUNS", "PORTLAND TRAIL BLAZERS", "SACRAMENTO KINGS",
    "SAN ANTONIO SPURS", "UTAH JAZZ"
]


def find_ID_by_name(team_name: str) -> int:
    """Retorna o TEAM_ID dado o nome completo do time."""
    info = teams.find_teams_by_full_name(team_name)
    return info[0]["id"]

def find_abb_by_name(team_name: str) -> str:
    """Retorna a abreviação do time (GSW, LAL...)."""
    info = teams.find_teams_by_full_name(team_name)
    return info[0]["abbreviation"]

def getting_ID_row_by_name(team_list, df_teams):
    """Adiciona TEAM_ID no DF a partir do nome do time."""
    mapping = {team: find_ID_by_name(team) for team in team_list}
    df_teams["TEAM_ID"] = df_teams["TEAMS"].map(mapping)
    return df_teams

# Busca de parâmetros

In [ ]:

#Ofensive stats
def get_ofensive_stats(season: str, season_type: str) -> pd.DataFrame:
    advanced = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        measure_type_detailed_defense='Advanced',
        per_mode_detailed="PerGame",
        season_type_all_star=season_type
    ).get_data_frames()[0]

    cols_adv = [
        'TEAM_ID', 'OFF_RATING', 'NET_RATING', 'AST_PCT',
        'AST_TO', 'AST_RATIO', 'TM_TOV_PCT', 'EFG_PCT',
        'TS_PCT', 'PACE', 'E_PACE'
    ]

    four_factors = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        measure_type_detailed_defense='Four Factors',
        per_mode_detailed="PerGame",
        season_type_all_star=season_type
    ).get_data_frames()[0]

    cols_ff = ['TEAM_ID', 'FTA_RATE', 'OREB_PCT']

    df = pd.merge(advanced[cols_adv], four_factors[cols_ff], on="TEAM_ID")
    return df

#Defensive stats
def get_defensive_stats(season: str, season_type: str) -> pd.DataFrame:
    defense = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        measure_type_detailed_defense='Defense',
        per_mode_detailed="PerGame",
        season_type_all_star=season_type
    ).get_data_frames()[0]

    cols_def = [
        'TEAM_ID', 'DEF_RATING', 'STL', 'BLK',
        'DREB_PCT', 'OPP_PTS_2ND_CHANCE', 'OPP_PTS_PAINT'
    ]

    opponent = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        measure_type_detailed_defense='Opponent',
        per_mode_detailed="PerGame",
        season_type_all_star=season_type
    ).get_data_frames()[0]

    cols_op = [
        'TEAM_ID', 'OPP_FGM', 'OPP_FGA', 'OPP_FG_PCT',
        'OPP_FG3M', 'OPP_FG3A', 'OPP_FG3_PCT', 'OPP_FTA',
        'OPP_REB', 'OPP_AST', 'OPP_TOV', 'OPP_PTS'
    ]

    df = pd.merge(defense[cols_def], opponent[cols_op], on='TEAM_ID')
    return df



#Bench points%
def get_bench_points(season: str, season_type: str) -> pd.DataFrame:
    bench = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        measure_type_detailed_defense='Base',
        starter_bench_nullable='Bench',
        season_type_all_star=season_type
    ).get_data_frames()[0]

    team = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        measure_type_detailed_defense='Base',
        season_type_all_star=season_type
    ).get_data_frames()[0]

    bench = bench[['TEAM_ID', 'PTS']].rename(columns={'PTS': 'BENCH_PTS'})
    team = team[['TEAM_ID', 'PTS']]

    df = pd.merge(bench, team, on='TEAM_ID')
    df['BENCH_PTS_PCT'] = df['BENCH_PTS'] / df['PTS']

    return df[['TEAM_ID', 'BENCH_PTS_PCT']]

#Shot profile/ Team identity
def get_shot_locations(season: str, season_type: str) -> pd.DataFrame:
    scoring = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        measure_type_detailed_defense='Scoring',
        per_mode_detailed="PerGame",
        season_type_all_star=season_type
    ).get_data_frames()[0]

    shot = leaguedashteamshotlocations.LeagueDashTeamShotLocations(
        season=season,
        measure_type_simple='Base',
        per_mode_detailed="PerGame",
        season_type_all_star=season_type
    ).get_data_frames()[0]

    fgm_3 = (
        shot['Left Corner 3', 'FGM'] +
        shot['Right Corner 3', 'FGM'] +
        shot['Above the Break 3', 'FGM']
    )

    fga_3 = (
        shot['Left Corner 3', 'FGA'] +
        shot['Right Corner 3', 'FGA'] +
        shot['Above the Break 3', 'FGA']
    )

    FG3_PCT = fgm_3 / fga_3

    df = pd.DataFrame()
    df["TEAM_ID"] = shot[("", "TEAM_ID")]
    df["RIM_PCT"] = shot["Restricted Area", "FG_PCT"]
    df["MID_PCT"] = shot["Mid-Range", "FG_PCT"]
    df["FG3_PCT"] = FG3_PCT
    df["PCT_PTS_2PT"] = scoring["PCT_PTS_2PT"]
    df["PCT_PTS_3PT"] = scoring["PCT_PTS_3PT"]

    return df

def get_usage_stars(season: str, season_type: str, min_games: int = 30) -> pd.DataFrame:
    df = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_type_all_star=season_type,
        per_mode_detailed='PerGame',
        measure_type_detailed_defense='Advanced'
    ).get_data_frames()[0]

    df = df[["PLAYER_NAME", "TEAM_ID", "GP", "USG_PCT"]]
    df = df[df['GP'] >= min_games]

    result = []

    for tid in df["TEAM_ID"].unique():
        team_df = df[df["TEAM_ID"] == tid].sort_values(by="USG_PCT", ascending=False).head(2)
        if len(team_df) == 2:
            result.append({
                "TEAM_ID": tid,
                "LEADER_1": team_df.iloc[0]["PLAYER_NAME"],
                "USG_L1": round(team_df.iloc[0]["USG_PCT"], 1),
                "LEADER_2": team_df.iloc[1]["PLAYER_NAME"],
                "USG_L2": round(team_df.iloc[1]["USG_PCT"], 1)
            })

    return pd.DataFrame(result)
